In [ ]:
# Multi-Mine Production Forecasting Analysis

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import pmdarima as pm
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

# List of mines to analyze
MINES = ['Carlin', 'Cortez', 'Turquoise Ridge', 'Pueblo Viejo', 'Loulo-Gounkoto', 'Kibali', 'North Mara', 'Bulyanhulu']

# Realization factor (from Carlin analysis)
REALIZATION_FACTOR = 0.91

# Forecast periods (20 quarters = 5 years)
N_PERIODS = 20

# Conversion factor: 1 tonne = 31.1034768 troy ounces
TROY_OZ_CONVERSION = 1 / 31.1034768

# Random seed for reproducibility
np.random.seed(42)

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def convert_quarter_to_date(q_str):
    """Convert quarter string (e.g., 'Q1 2019') to datetime"""
    year = q_str.split(' ')[1]
    quarter = q_str.split(' ')[0]
    
    if quarter == 'Q1':
        return f"{year}-03-31"
    elif quarter == 'Q2':
        return f"{year}-06-30"
    elif quarter == 'Q3':
        return f"{year}-09-30"
    else:  # Q4
        return f"{year}-12-31"

def prepare_mine_data(df, mine_name):
    """Prepare time series data for a specific mine"""
    mine_data = df[df['Nome miniera'] == mine_name].copy()
    
    mine_data['date_time'] = pd.to_datetime(
        mine_data['data'].apply(convert_quarter_to_date)
    )
    
    data_ready = mine_data.set_index('date_time').sort_index()
    data_ready = data_ready.drop(columns=['data', 'Nome miniera'])
    data_ready = data_ready.asfreq('QE')
    
    first_valid_idx = data_ready.apply(lambda col: col.first_valid_index())
    
    if first_valid_idx.notna().any():
        start_idx = first_valid_idx.max()
        data_ready = data_ready.loc[start_idx:]
        print(f"   ℹ️  Removed leading NaN rows. Data starts from: {start_idx.date()}")
    
    return data_ready

def forecast_with_sarima(series, n_periods=20, order=(1,0,0), seasonal_order=(1,0,0,4)):
    """Fit a targeted SARIMA model based on operational logic"""
    try:
        model = pm.ARIMA(
            order=order,
            seasonal_order=seasonal_order,
            suppress_warnings=True,
            out_of_sample_size=0 # Ensures all small data is used for training
        )
        
        model.fit(series)
        
        forecast, conf_int = model.predict(
            n_periods=n_periods,
            return_conf_int=True,
            alpha=0.20 # 80% CI
        )
        
        return forecast, conf_int, model
        
    except Exception as e:
        print(f"   ❌ Error in SARIMA fitting: {e}")
        return None, None, None

def simulate_recovery_rate(historical_data, n_periods=20):
    """Simulate recovery rate using uniform distribution based on historical range"""
    min_rate = historical_data.min()
    max_rate = historical_data.max()
    
    min_rate = max(min_rate - 0.01, 0.5)
    max_rate = min(max_rate + 0.01, 1.0)
    
    simulated = np.random.uniform(min_rate, max_rate, n_periods)
    return simulated.round(2)

def calculate_production_forecast(forecast_df, ore_col, grade_col, recovery_col, 
                                  ci_ore_lower, ci_ore_upper, 
                                  ci_grade_lower, ci_grade_upper,
                                  recovery_min, recovery_max,
                                  realization_factor, alpha):
    """
    Calculate the production forecast by applying the exact variance formula
    for the product of independent variables.
    """
    z_score = stats.norm.ppf(1 - alpha/2)
    
    base_production = (
        forecast_df[ore_col] * forecast_df[grade_col] * forecast_df[recovery_col] * TROY_OZ_CONVERSION * realization_factor
    )
    
    sigma_ore = (forecast_df[ci_ore_upper] - forecast_df[ore_col]) / z_score
    r_ore = sigma_ore / forecast_df[ore_col]
    
    sigma_grade = (forecast_df[ci_grade_upper] - forecast_df[grade_col]) / z_score
    r_grade = sigma_grade / forecast_df[grade_col]
    
    mean_recovery = forecast_df[recovery_col].mean()
    sigma_recovery = (recovery_max - mean_recovery) / z_score
    r_recovery = sigma_recovery / mean_recovery

    r_prod_sq = (1 + r_ore**2) * (1 + r_grade**2) * (1 + r_recovery**2) - 1
    r_prod = np.sqrt(r_prod_sq)
    
    forecast_df['Quarterly_Std_Dev'] = base_production * r_prod
    forecast_df['Quarterly_Variance'] = forecast_df['Quarterly_Std_Dev'] ** 2

    forecast_df['Forecast Production'] = base_production.round(0).astype(int)
    forecast_df['CI Production Lower'] = (base_production - (z_score * forecast_df['Quarterly_Std_Dev'])).clip(lower=0).round(0).astype(int)
    forecast_df['CI Production Upper'] = (base_production + (z_score * forecast_df['Quarterly_Std_Dev'])).round(0).astype(int)
    
    # ========================================================================
    # ANNUAL SUMMARY
    # ========================================================================
    forecast_df['Year'] = forecast_df.index.year
    
    # sum of production and variance for each year
    annual_df = forecast_df.groupby('Year').agg({
        'Forecast Production': 'sum',
        'Quarterly_Variance': 'sum'
    })
    
    # standard deviation of the annual production is the square root of the sum of quarterly variances
    annual_df['Annual_Std_Dev'] = np.sqrt(annual_df['Quarterly_Variance'])
    
    annual_df['CI Production Lower'] = (
        annual_df['Forecast Production'] - (z_score * annual_df['Annual_Std_Dev'])
    ).clip(lower=0).round(0).astype(int)
    
    annual_df['CI Production Upper'] = (
        annual_df['Forecast Production'] + (z_score * annual_df['Annual_Std_Dev'])
    ).round(0).astype(int)
    
    # Keep only relevant columns for annual summary
    annual_df = annual_df[['Forecast Production', 'CI Production Lower', 'CI Production Upper']]
    forecast_df = forecast_df.drop(columns=['Quarterly_Std_Dev', 'Quarterly_Variance'])
    
    return forecast_df, annual_df

# ============================================================================
# MAIN ANALYSIS FUNCTION
# ============================================================================

def analyze_mine(initial_data, mine_name, n_periods=20):
    """Complete analysis pipeline for a single mine"""
    print(f"\n{'='*80}")
    print(f"ANALYZING: {mine_name}")
    print(f"{'='*80}\n")
    
    data_ready = prepare_mine_data(initial_data, mine_name)
    
    if len(data_ready) < 8:
        print(f"⚠️  Insufficient data for {mine_name} (only {len(data_ready)} quarters)")
        return None
    
    print(f"Historical data: {len(data_ready)} quarters")
    print(f"Date range: {data_ready.index[0].date()} to {data_ready.index[-1].date()}\n")
    
    # 1. FORECAST ORE PROCESSED (Stationary, Seasonal, Memory)
    print("1️⃣  Forecasting Ore Processed...")
    forecast_ore, conf_int_ore, model_ore = forecast_with_sarima(
        data_ready['ore processed'], 
        n_periods=n_periods,
        order=(1, 0, 1), 
        seasonal_order=(1, 0, 0, 4)
    )
    
    if forecast_ore is None:
        print(f"   ❌ Failed to forecast ore processed for {mine_name}")
        return None
    print(f"   Model: {model_ore.order} x {model_ore.seasonal_order}")
    
    # 2. FORECAST AVERAGE GRADE (Non-Stationary, Non-Seasonal, Trending)
    print("2️⃣  Forecasting Average Grade...")
    forecast_grade, conf_int_grade, model_grade = forecast_with_sarima(
        data_ready['average grade'], 
        n_periods=n_periods,
        order=(1, 0, 0), 
        seasonal_order=(0, 0, 0, 0) # Strictly no seasonality
    )
    
    if forecast_grade is None:
        print(f"   ❌ Failed to forecast average grade for {mine_name}")
        return None
    print(f"   Model: {model_grade.order} x {model_grade.seasonal_order}")
    
    # 3. SIMULATE RECOVERY RATE
    print("3️⃣  Simulating Recovery Rate...")
    recovery_simulation = simulate_recovery_rate(
        data_ready['recovery rate'], n_periods
    )
    recovery_min = data_ready['recovery rate'].min()
    recovery_max = data_ready['recovery rate'].max()
    print(f"   Range: {recovery_min:.2f} - {recovery_max:.2f}")
    
    # 4. BUILD FORECAST DATAFRAME
    forecast_index = pd.date_range(
        start=data_ready.index[-1] + pd.DateOffset(months=3),
        periods=n_periods,
        freq='QE'
    )
    
    forecast_df = pd.DataFrame(index=forecast_index)
    forecast_df['Forecast Ore Processed'] = forecast_ore.round(0).astype(int)
    forecast_df['CI Ore Processed Lower'] = conf_int_ore[:, 0].round(0).astype(int)
    forecast_df['CI Ore Processed Upper'] = conf_int_ore[:, 1].round(0).astype(int)
    
    forecast_df['Forecast Average Grade'] = forecast_grade.round(2)
    forecast_df['CI Average Grade Lower'] = conf_int_grade[:, 0].round(2)
    forecast_df['CI Average Grade Upper'] = conf_int_grade[:, 1].round(2)
    
    forecast_df['Forecast Recovery Rate'] = recovery_simulation
    
    # 5 & 6. CALCULATE PRODUCTION & ANNUAL SUMMARY
    print("4️⃣  Calculating Production Forecast and Annual Intervals...")
    forecast_df, annual_production = calculate_production_forecast(
        forecast_df,
        'Forecast Ore Processed',
        'Forecast Average Grade',
        'Forecast Recovery Rate',
        'CI Ore Processed Lower',
        'CI Ore Processed Upper',
        'CI Average Grade Lower',
        'CI Average Grade Upper',
        recovery_min,
        recovery_max,
        REALIZATION_FACTOR,
        alpha=0.2
    )
    
    print("\n📊 Annual Production Forecast (oz):")
    print(annual_production)

    return {
        'mine_name': mine_name,
        'historical_data': data_ready,
        'forecast': forecast_df,
        'annual_production': annual_production,
        'models': {
            'ore': model_ore,
            'grade': model_grade
        }
    }

# ============================================================================
# VISUALIZATION FUNCTION
# ============================================================================

def plot_mine_forecast(result):
    """Create comprehensive visualization for mine forecast"""
    if result is None:
        return
    
    mine_name = result['mine_name']
    historical = result['historical_data']
    forecast = result['forecast']
    
    fig, axes = plt.subplots(2, 2, figsize=(10, 5))
    fig.suptitle(f'{mine_name} - Production Forecast Analysis', 
                 fontsize=16, fontweight='bold')
    
    ax1 = axes[0, 0]
    ax1.plot(historical.index, historical['ore processed'], 
             label='Historical', color='blue', linewidth=2)
    ax1.plot(forecast.index, forecast['Forecast Ore Processed'], 
             label='Forecast', color='red', linestyle='--', linewidth=2)
    ax1.fill_between(forecast.index, 
                      forecast['CI Ore Processed Lower'],
                      forecast['CI Ore Processed Upper'],
                      alpha=0.2, color='red', label='80% CI')
    ax1.set_title('Ore Processed (tonnes)')
    ax1.set_xlabel('Date')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    ax2 = axes[0, 1]
    ax2.plot(historical.index, historical['average grade'], 
             label='Historical', color='green', linewidth=2)
    ax2.plot(forecast.index, forecast['Forecast Average Grade'], 
             label='Forecast', color='darkgreen', linestyle='--', linewidth=2)
    ax2.fill_between(forecast.index,
                      forecast['CI Average Grade Lower'],
                      forecast['CI Average Grade Upper'],
                      alpha=0.2, color='green', label='80% CI')
    ax2.set_title('Average Grade (g/t)')
    ax2.set_xlabel('Date')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    ax3 = axes[1, 0]
    ax3.plot(historical.index, historical['recovery rate'], 
             label='Historical', color='purple', linewidth=2)
    ax3.plot(forecast.index, forecast['Forecast Recovery Rate'], 
             label='Simulated', color='magenta', linestyle='--', 
             linewidth=2, marker='o', markersize=3)
    ax3.set_title('Recovery Rate (%)')
    ax3.set_xlabel('Date')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    ax4 = axes[1, 1]
    ax4.plot(forecast.index, forecast['Forecast Production'], 
             label='Forecast Production', color='orange', 
             linewidth=2, marker='o', markersize=4)
    ax4.fill_between(forecast.index,
                      forecast['CI Production Lower'],
                      forecast['CI Production Upper'],
                      alpha=0.3, color='orange', label='80% CI')
    ax4.set_title('Production Forecast (oz)')
    ax4.set_xlabel('Date')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# ============================================================================
# MAIN EXECUTION
# ============================================================================

print("""
╔════════════════════════════════════════════════════════════════════════════╗
║                   MULTI-MINE PRODUCTION FORECASTING                        ║
║                                                                            ║
║  This script analyzes multiple mines using SARIMA forecasting for          ║
║  ore processed and average grade, uniform distribution for recovery        ║
║  rate, and calculates production forecasts with confidence intervals.      ║
╚════════════════════════════════════════════════════════════════════════════╝
""")

# Load data
# ---- USER INPUT: path to the Barrick quarterly data file ----
DATA_PATH = "data/Quarterly Data.xlsx"

initial_data = pd.read_excel(DATA_PATH, sheet_name='Dati_P')

all_results = {}

for mine_name in MINES:
    result = analyze_mine(initial_data, mine_name, N_PERIODS)
    if result is not None:
        all_results[mine_name] = result
        plot_mine_forecast(result)

# Create summary comparison
print("\n" + "="*80)
print("SUMMARY: Total Annual Production by Mine (000s ozs)")
print("="*80)

summary_data = []
for mine_name, result in all_results.items():
    if result is None:
        continue
    
    annual = result['annual_production']
    for year, row in annual.iterrows():
        summary_data.append({
            'Mine': mine_name,
            'Year': year,
            'CIlow': row['CI Production Lower'],
            'Production': row['Forecast Production'],
            'CIup': row['CI Production Upper']
        })

if len(summary_data) == 0:
    print("⚠️ ERROR: No forecast data was successfully generated for any mine.")
    print("👉 Check the logs above to identify data or model issues.")
else:
    summary_df = pd.DataFrame(summary_data)
    
    summary_pivot = summary_df.pivot(
        index='Year', 
        columns='Mine', 
        values=['CIlow', 'Production', 'CIup']
    )
    
    summary_pivot_prod = summary_df.pivot(
        index='Year', 
        columns='Mine', 
        values='Production'
    )

    print("\nTotal Production Forecast by Mine (000s ozs):")
    print(summary_pivot_prod.round(1))
    
    # Reorder columns -> (CI Lower, Forecast, CI Upper)
    summary_pivot = summary_pivot.swaplevel(0, 1, axis=1).sort_index(axis=1)

    print("Total Production Across All Mines (000s ozs):")
    total_production = summary_pivot.groupby(level=1, axis=1).sum()
    total_production = total_production[['CIlow', 'Production', 'CIup']]
    print(total_production)


In [ ]:
correlation_matrix = initial_data.iloc[:, 2:5].corr()
sns.reset_orig()
plt.style.use('default')
plt.figure(figsize=(8, 4))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
plt.show()

In [ ]:
# ============================================================================
# HISTORICAL PRODUCTION VISUALIZATION (ALL MINES)
# ============================================================================

print("\n" + "="*80)
print("GENERATING HISTORICAL PRODUCTION GRAPH...")
print("="*80)

# Initialize the plot
plt.figure(figsize=(10, 5))

# Loop through each mine to calculate and plot historical production
for mine_name in MINES:
    # Fetch the cleaned historical data using your existing helper function
    hist_data = prepare_mine_data(initial_data, mine_name)
    
    if hist_data is not None and len(hist_data) > 0:
        # Calculate historical production accurately
        historical_production = (
            hist_data['ore processed'] * hist_data['average grade'] * hist_data['recovery rate'] * TROY_OZ_CONVERSION * REALIZATION_FACTOR
        )
        
        # Plot the line for this specific mine
        plt.plot(
            hist_data.index, 
            historical_production, 
            label=mine_name, 
            linewidth=2, 
            marker='o', 
            markersize=4
        )

# Formatting the graph
plt.title('Historical Gold Production by Mine (Up to 2025)', fontsize=18, fontweight='bold')
plt.xlabel('Quarter', fontsize=12, fontweight='bold')
plt.ylabel('Gold Production (Troy Ounces)', fontsize=12, fontweight='bold')

# Add a grid for easier reading
plt.grid(True, linestyle='--', alpha=0.6)

# Place the legend outside the plot so it doesn't cover the data lines
plt.legend(title='Barrick Mines', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)

# Automatically adjust layout so the legend isn't cut off
plt.tight_layout()

# Display the graph
plt.show()

In [ ]:
# ============================================================================
# TOTAL COMPANY PRODUCTION VISUALIZATION (HISTORICAL + FORECAST)
# ============================================================================

print("\n" + "="*80)
print("GENERATING TOTAL COMPANY PRODUCTION GRAPH...")
print("="*80)

# 1. Aggregate Total Historical Production
hist_frames = []
for mine_name in MINES:
    hist_data = prepare_mine_data(initial_data, mine_name)
    if hist_data is not None and len(hist_data) > 0:
        hist_prod = (
            hist_data['ore processed'] * hist_data['average grade'] * hist_data['recovery rate'] * TROY_OZ_CONVERSION * REALIZATION_FACTOR
        )
        hist_prod.name = mine_name
        hist_frames.append(hist_prod)

# Concatenate all historical series and sum them by date
total_historical = pd.concat(hist_frames, axis=1).sum(axis=1)

# 2. Aggregate Total Forecasted Production
forecast_frames = []
lower_frames = []
upper_frames = []

for mine_name, result in all_results.items():
    if result is not None:
        f_df = result['forecast']
        forecast_frames.append(f_df['Forecast Production'].rename(mine_name))
        lower_frames.append(f_df['CI Production Lower'].rename(mine_name))
        upper_frames.append(f_df['CI Production Upper'].rename(mine_name))

# Concatenate all forecast series and sum them by date
total_forecast = pd.concat(forecast_frames, axis=1).sum(axis=1)
total_lower = pd.concat(lower_frames, axis=1).sum(axis=1)
total_upper = pd.concat(upper_frames, axis=1).sum(axis=1)

# 3. Create the Visualization
plt.figure(figsize=(10, 5))

# Plot the combined historical line
plt.plot(
    total_historical.index, 
    total_historical, 
    label='Total Historical Production', 
    color='blue', 
    linewidth=2.5, 
    marker='o',
    markersize=4
)

# Plot the combined forecast line
plt.plot(
    total_forecast.index, 
    total_forecast, 
    label='Total Forecasted Production', 
    color='orange', 
    linestyle='--', 
    linewidth=2.5, 
    marker='s',
    markersize=4
)

# Add the aggregated confidence intervals
plt.fill_between(
    total_forecast.index, 
    total_lower, 
    total_upper, 
    color='orange', 
    alpha=0.25, 
    label='80% Confidence Interval (Aggregated)'
)

# Formatting the graph
plt.title('Barrick Gold: Total Company Production (Historical vs. Forecast)', fontsize=16, fontweight='bold')
plt.xlabel('Date (Quarterly)', fontsize=12, fontweight='bold')
plt.ylabel('Gold Production (Troy Ounces)', fontsize=12, fontweight='bold')

plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='upper right', fontsize=11)
plt.tight_layout()

# Display the graph
plt.show()

In [ ]:
# ============================================================================
# TOTAL COMPANY PRODUCTION VISUALIZATION (ANNUAL HISTORICAL VS FORECAST)
# ============================================================================

print("\n" + "="*80)
print("GENERATING TOTAL COMPANY PRODUCTION GRAPH (ANNUAL)...")
print("="*80)

# 1. Aggregate Exact Historical Production by Year
hist_frames = []
for mine_name in MINES:
    hist_data = prepare_mine_data(initial_data, mine_name)
    if hist_data is not None and len(hist_data) > 0:
        # Calculate historical production accurately based on your factors
        hist_prod = (
            hist_data['ore processed'] * hist_data['average grade'] * hist_data['recovery rate'] * TROY_OZ_CONVERSION * REALIZATION_FACTOR
        )
        hist_prod.name = mine_name
        hist_frames.append(hist_prod)

# Sum all mines quarterly, then group into annual totals
total_historical_q = pd.concat(hist_frames, axis=1).sum(axis=1)
total_historical_annual = total_historical_q.groupby(total_historical_q.index.year).sum()

# Convert to 000s of Ounces for the plot (matching the summary table scale)
total_historical_annual = total_historical_annual / 1000.0

# 2. Extract Forecast Data and Calculate Mathematically Exact Total CIs
z_score = stats.norm.ppf(1 - 0.20/2) # Exact Z-score for 80% CI
variance_annual_totals = None
forecast_annual_totals = None

for mine_name, result in all_results.items():
    if result is not None:
        ann_df = result['annual_production']
        
        # Initialize the series on the first valid mine
        if forecast_annual_totals is None:
            forecast_annual_totals = ann_df['Forecast Production'].copy()
            # Reverse-engineer the variance from the upper bound to aggregate it correctly
            variance_annual_totals = ((ann_df['CI Production Upper'] - ann_df['Forecast Production']) / z_score) ** 2
        else:
            forecast_annual_totals += ann_df['Forecast Production']
            # Variances can be added linearly across independent mines
            variance_annual_totals += ((ann_df['CI Production Upper'] - ann_df['Forecast Production']) / z_score) ** 2

# Calculate the precise Company-Wide Standard Deviation and Confidence Intervals
company_annual_std = np.sqrt(variance_annual_totals)
total_lower_annual = forecast_annual_totals - (z_score * company_annual_std)
total_upper_annual = forecast_annual_totals + (z_score * company_annual_std)

# Scale all forecast metrics to 000s of Ounces
forecast_plot = forecast_annual_totals / 1000.0
lower_plot = total_lower_annual / 1000.0
upper_plot = total_upper_annual / 1000.0

# 3. Generate the Visualization
plt.figure(figsize=(14, 7))

# Plot Historical Data
plt.plot(
    total_historical_annual.index, 
    total_historical_annual, 
    label='Historical Total', 
    color='blue', 
    linewidth=3, 
    marker='o',
    markersize=6
)

# Plot Forecasted Data
plt.plot(
    forecast_plot.index, 
    forecast_plot, 
    label='Forecast Total', 
    color='orange', 
    linestyle='--', 
    linewidth=3, 
    marker='s',
    markersize=6
)

# Plot the Exact Confidence Intervals
plt.fill_between(
    forecast_plot.index, 
    lower_plot, 
    upper_plot, 
    color='orange', 
    alpha=0.25, 
    label='80% CI (Exact Variance Aggregation)'
)

# Formatting
plt.title('Barrick Gold: Total Annual Production (Historical vs. Forecast)', fontsize=16, fontweight='bold')
plt.xlabel('Year', fontsize=12, fontweight='bold')
plt.ylabel('Production (000s Troy Ounces)', fontsize=12, fontweight='bold')

# Force X-axis to display clean integer years
all_years = list(total_historical_annual.index) + list(forecast_plot.index)
plt.xticks(np.arange(min(all_years), max(all_years) + 1, 1))

plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='lower left', fontsize=11)
plt.tight_layout()

# Render the plot
plt.show()